In [ ]:
#| default_exp game/web

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import httpx
import random
from starlette.responses import Response

In [ ]:
#| export
import logging

for logger_name in ("uvicorn", "uvicorn.error", "uvicorn.access"):
    uv_logger = logging.getLogger(logger_name)
    file_handler = logging.FileHandler('HexServer.txt')
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    uv_logger.addHandler(file_handler)


In [ ]:
#| export
daisy_hdrs = (
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css')
)

In [ ]:
#| export
app = FastHTML(hdrs=daisy_hdrs)
rt = app.route

In [ ]:
#| export
def get_preview(app):
    return partial(HTMX, app=app, host=None, port=None)
preview = get_preview(app)

In [ ]:
srv = JupyUvi(app=app)

In [ ]:
#| export
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion , windy_edge , unique_windy_edge
from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain
Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins, Watershed

In [ ]:
#| export
from HexMagic.game.kingdom import GameBoard,Kingdom,TradeRoute,Terrain,StyleCSS, Hex, TerraDemo, Geology, DrainageBasins, CountryFlag
from HexMagic.game.country import CountryDetails
from HexMagic.game.store import HexServer, DatabaseDebugger

with DatabaseDebugger(keep_on_error=False) as debugger:
    debugger.run_all()

In [ ]:
#| export
# Lets create a server
mainServer = HexServer()

In [ ]:
#| export
import logging

logging.basicConfig(
    filename='base.text',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logging.info("getting Started")


In [ ]:
!tail base.text

2026-02-01 16:03:14,189 - INFO - layer[6] is block
2026-02-01 16:03:14,189 - INFO - layer[7] is hexes
2026-02-01 16:03:50,176 - INFO - layer[0] is terrain_base
2026-02-01 16:03:50,176 - INFO - layer[1] is climates
2026-02-01 16:03:50,177 - INFO - layer[2] is settlement
2026-02-01 16:03:50,177 - INFO - layer[3] is countries
2026-02-01 16:03:50,177 - INFO - layer[4] is water
2026-02-01 16:03:50,177 - INFO - layer[5] is names
2026-02-01 16:03:50,177 - INFO - layer[6] is hexes
2026-02-01 23:43:35,687 - INFO - getting Started


## Map Helpers

In [ ]:
#| export
@patch
def html(self:Terrain,wrapper:HexWrapper = None)->str:
    grid = self.hexGrid
    if wrapper is None:
        wrapper = HexWrapper(callBack=HexWrapper.route())
    clearStyle = StyleCSS("HexClear",stroke="none",fill="white",opacity=0.1)
    for i, h in enumerate(grid.hexes):
        grid.hexes[i].style = clearStyle
        #grid.hexes[i].label = str(i)
    #self.colorMap()
    grid.builder.add_style(clearStyle)
    grid.update(wrapper=wrapper,layer_name="hexes")
    for i, l in enumerate(grid.builder.layers):
        logging.info(f"layer[{i}] is {l.name}")
    return grid.builder.xml()

In [ ]:
#| export
def get_board(session):
    """Get active board for session, returns None if no game exists."""
    result = mainServer.load_game_by_session(session['userid'])
    return result.data if result.status == 'loaded' else None


In [ ]:

def countryTerrain(board,countryId):
    countries = board.kingdoms
    showCountry = countries[countryId - 1]
    text = ""
    
    mapDetails  = CountryDetails(showCountry)  
    terrain = mapDetails.countryMap

    grid = terrain.hexGrid
    builder = grid.builder

    grid.adjustRadius(20)

    builder.layers = []

    terrain.colorMap()
    terrain.compute_climate()
    
    terrain.terrainCream()

    

    builder.adjust("climates", terrain.dottedClimate())
    builder.adjust("settlement",mapDetails.settlements_overlay(countries))
    
    builder.adjust("countries",mapDetails.countries_overlay(countries))

    basins = DrainageBasins(terrain)
    #builder.adjust("countries", board.countries_overlay())
    builder.adjust("water", basins.draw_watersheds(max_width=4))
    builder.adjust("names",mapDetails.names_overlay(countries))
    builder.adjust("block",mapDetails.unknownOverlay())
    #wrapper = HexWrapper(callBack=HexWrapper.route())
    detailWrapper = HexWrapper(callBack = lambda grid,index :  {
    "hx-post": f"/country_hex_clicked", #The post gives the route back to the url
    "hx-vals": f'{{"hex_id":{index},"country_id":{showCountry.countryId}}}', # the parameter needs to be called the same in the route
    "hx-target": f"#map" # the element we need to update
    })

    text = NotStr(terrain.html(wrapper=detailWrapper))
    return text


## serving Maps

In [ ]:
#| export
@rt
def newgame(session):
    """Landing page to create a new game from a template."""
    if 'userid' not in session: 
        session['userid'] = random.randint(0, 1_000_000)
    
    templates = mainServer.list_templates()
    
    return Titled("New Game",
        Form(
            Select(
                *[Option(name, value=tid) for tid, name in templates],
                name="template_id", cls="select select-bordered w-full max-w-xs"
            ),
            Input(name="game_name", placeholder="Game name (optional)", 
                  cls="input input-bordered w-full max-w-xs mt-2"),
            Input(name="top_n", type="number", value="5", 
                  placeholder="Number of kingdoms", cls="input input-bordered w-full max-w-xs mt-2"),
            Button("Create Game", type="submit", cls="btn btn-primary mt-4"),
            action="/create_game", method="post"
        )
    )


In [ ]:
#| export
@rt
def create_game(session, template_id: int, game_name: str = None, top_n: int = 5):
    """Create a new game and redirect to the map."""
    if 'userid' not in session:
        session['userid'] = random.randint(0, 1_000_000)
    
    result = mainServer.start_from_template(
        session['userid'], template_id, game_name or None, 
        top_n=top_n, rounds=50
    )
    
    if result.status == 'created':
        return RedirectResponse('/showMap', status_code=303)
    else:
        return P(f"Error: {result.context}")


In [ ]:
#| export
@rt
def showMap(session):
    """Show the map, or redirect to new game if none exists."""
    if 'userid' not in session: 
        session['userid'] = random.randint(0, 1_000_000)
    
    board = get_board(session)
    if board is None:
        return RedirectResponse('/newgame', status_code=303)
    
    terrain = board.terrain
    grid = terrain.hexGrid
    builder = grid.builder

    terrain.colorMap()
    grid.update()
    terrain.compute_climate()

    builder.layers = []
    terrain.terrainCream()

    builder.adjust("climates", terrain.dottedClimate())
    builder.adjust("settlement", board.settlements_overlay())
    builder.adjust("countries", board.countries_overlay())
    builder.adjust("water", board.world.basins.draw_watersheds())
    builder.adjust("names", board.names_overlay())

    mapText = P(NotStr(terrain.html()), id="map")
    return Titled("Hex Map",
        Div(
            Div(H4("Menu", cls="text-lg font-bold mb-4")),
            Div(
                Div(mapText, id="map-area", cls="p-4 overflow-auto flex-1"),
                cls="flex-1 flex flex-col overflow-hidden", height=800
            ),
            cls="flex h-screen"
        )
    )


In [ ]:
#| export

@rt
def debugMap(session):
    # ... existing session/board checks ...
    
    board = get_board(session)
    if board is None:
        return RedirectResponse('/newgame', status_code=303)
    
    encoded = board.terrain.encode()
    return Response(
        content=encoded,
        media_type="text/plain",
        headers={"Content-Disposition": "attachment; filename=terrain.txt"}
    )


In [ ]:
#| export
@rt
def svgMap(session):
    board = get_board(session)
    if board is None:
        return RedirectResponse('/newgame', status_code=303)
    
    # Set up the terrain/grid as in showMap
    terrain = board.terrain
    grid = terrain.hexGrid
    builder = grid.builder
    
    terrain.colorMap()
    grid.update()
    terrain.compute_climate()
    builder.layers = []
    terrain.terrainCream()
    # Add any other layers you want...

    builder.adjust("climates", terrain.dottedClimate())
    builder.adjust("settlement", board.settlements_overlay())
    builder.adjust("countries", board.countries_overlay())
    builder.adjust("water", board.world.basins.draw_watersheds())
    builder.adjust("names", board.names_overlay())
    
    return Response(
        content=terrain.html(),
        media_type="image/svg+xml",
        headers={"Content-Disposition": "attachment; filename=map.svg"}
    )


can I do something similar called svgMap that returns the svb filer ie the raw html file with a content type of svg

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! You can return raw SVG by using `Response` with `media_type="image/svg+xml"`. Something like:

```python
@rt
def svgMap(session):
    board = get_board(session)
    if board is None:
        return RedirectResponse('/newgame', status_code=303)
    
    # Set up the terrain/grid as in showMap
    terrain = board.terrain
    grid = terrain.hexGrid
    builder = grid.builder
    
    terrain.colorMap()
    grid.update()
    terrain.compute_climate()
    builder.layers = []
    terrain.terrainCream()
    # Add any other layers you want...
    
    return Response(
        content=terrain.html(),
        media_type="image/svg+xml"
    )
```

The key difference from `debugMap` is using `"image/svg+xml"` as the media type. This tells the browser to render it as an SVG image directly.

If you want the browser to download it as a file instead of displaying it, add the `Content-Disposition` header like you did in `debugMap`.

In [ ]:
#| export
@rt
def country(session, country_id: int):
    board = get_board(session)
    if board is None:
        return RedirectResponse('/newgame', status_code=303)

    return countryTerrain(board,countryId)
        
   


In [ ]:
#| export
@rt
def debugCountry(session):
    board = get_board(session)
    if board is None:
        return RedirectResponse('/newgame', status_code=303)
    
    links = [A(k.name, href=f"/country/{k.id}") for k in board.kingdoms]

    return Ul(*[Li(link) for link in links])

## Handling clicks

In [ ]:
#| export
@rt
def hex_clicked(session, hex_id: int):
    "This does the raising of the map at a particular point"
    if 'userid' not in session: 
        session['userid'] = random.randint(0, 1_000_000)
    
    board = get_board(session)
    if board is None:
        return RedirectResponse('/newgame', status_code=303)

    terrain = board.terrain
    countries = terrain.fields["country"]
    countryId = int(countries[hex_id])

    if countryId == 0:
        return P(f"clicked {hex_id} which is water", id="map")
    elif countryId < 0:
        return P(f"clicked {hex_id} {countryId} which isn't habitable", id="map")
    else:
        return countryTerrain(board, countryId)


In [ ]:
#| export
@rt
def country_hex_clicked(session, hex_id: int, country_id: int):
    if 'userid' not in session: 
        session['userid'] = random.randint(0, 1_000_000)
    
    board = get_board(session)
    if board is None:
        return RedirectResponse('/newgame', status_code=303)

    countries = board.kingdoms
    if 0 < country_id <= len(countries):
        showCountry = countries[country_id - 1]
        mapDetails = CountryDetails(showCountry)
        for country in countries:
            destRegion = mapDetails.viewableRegion(country.region)
            if hex_id in destRegion:
                return countryTerrain(board, country.countryId)

    text = f"clicked {hex_id} which we couldn't resolve to a country"
    return Div(P(text, id="map"), A("back home", href="/showMap"))


## Demo the system

In [ ]:
#preview(newgame)

In [ ]:
preview(showMap)

In [ ]:
preview(debugMap)

In [ ]:
srv.stop()

In [ ]:
mainServer.reset_database()

So I want to start backing the web with a HexServer. We need to refactor how we get our map in show map and show country. we also need a landing page if a user does not have a gameboard. from there they would have a form  that they could use to create a new game board from a template 


**1. Initialize the HexServer** - we have a global `HexServer` mainServer

**2. Helper to get the board** - 

**3. Landing page with form** - route like `/newgame` that shows template choices and the `start_from_template` parameters

**4. Refactor `showMap`** - at the top, check if there's a board; if not, redirect to landing

can we start building the new from template page that is a form that redirects if we don't have a worl

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's start with the helper and the new game form. Here's a clean approach:

```python
def get_board(session):
    """Get active board for session, returns None if no game exists."""
    result = mainServer.load_game_by_session(session['userid'])
    return result.data if result.status == 'loaded' else None
```

And for the landing page form:

```python
@rt
def newgame(session):
    """Landing page to create a new game from a template."""
    if 'userid' not in session: 
        session['userid'] = random.randint(0, 1_000_000)
    
    templates = mainServer.list_templates()
    
    return Titled("New Game",
        Form(
            Select(
                *[Option(name, value=tid) for tid, name in templates],
                name="template_id", cls="select select-bordered w-full max-w-xs"
            ),
            Input(name="game_name", placeholder="Game name (optional)", 
                  cls="input input-bordered w-full max-w-xs mt-2"),
            Input(name="top_n", type="number", value="5", 
                  placeholder="Number of kingdoms", cls="input input-bordered w-full max-w-xs mt-2"),
            Button("Create Game", type="submit", cls="btn btn-primary mt-4"),
            action="/create_game", method="post"
        )
    )
```

Then the handler that processes the form:

```python
@rt
def create_game(session, template_id: int, game_name: str = None, top_n: int = 5):
    """Create a new game and redirect to the map."""
    if 'userid' not in session:
        session['userid'] = random.randint(0, 1_000_000)
    
    result = mainServer.start_from_template(
        session['userid'], template_id, game_name or None, 
        top_n=top_n, rounds=50
    )
    
    if result.status == 'created':
        return RedirectResponse('/showMap', status_code=303)
    else:
        return P(f"Error: {result.context}")
```

Does this structure look right? Once you've got these working, we can refactor `showMap` to use `get_board()` and redirect to `/newgame` when there's no active game.

can you refactor showMap

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's the refactored `showMap`:

```python
@rt
def showMap(session):
    """Show the map, or redirect to new game if none exists."""
    if 'userid' not in session: 
        session['userid'] = random.randint(0, 1_000_000)
    
    board = get_board(session)
    if board is None:
        return RedirectResponse('/newgame', status_code=303)
    
    terrain = board.terrain
    grid = terrain.hexGrid
    builder = grid.builder

    terrain.colorMap()
    grid.update()
    terrain.compute_climate()

    builder.layers = []
    terrain.terrainCream()

    builder.adjust("climates", terrain.dottedClimate())
    builder.adjust("settlement", board.settlements_overlay())
    builder.adjust("countries", board.countries_overlay())
    builder.adjust("water", board.world.basins.draw_watersheds())
    builder.adjust("names", board.names_overlay())

    mapText = P(NotStr(terrain.html()), id="map")
    return Titled("Hex Map",
        Div(
            Div(H4("Menu", cls="text-lg font-bold mb-4")),
            Div(
                Div(mapText, id="map-area", cls="p-4 overflow-auto flex-1"),
                cls="flex-1 flex flex-col overflow-hidden", height=800
            ),
            cls="flex h-screen"
        )
    )
```

Key changes:
- Uses `get_board(session)` instead of global `mainBoard`
- Redirects to `/newgame` if no board exists
- Replaced all `mainBoard` references with `board`

You'll also need to update `hex_clicked` and `countryHtml` similarly—they currently use the global `mainBoard`. Want me to show those refactors too?

can you sketch out the get_board helper

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's a `get_board` helper:

```python
def get_board(session):
    """Get board for current user. Returns (board, status)."""
    if 'userid' not in session:
        session['userid'] = str(random.randint(0, 1_000_000))
    
    server = HexServer()
    board, status = server.load_game_by_session(session['userid'])
    return board, status
```

And here's how you'd use it in `showMap`:

```python
@rt
def showMap(session):
    board, status = get_board(session)
    
    if status != 'loaded':
        return RedirectResponse('/newgame')
    
    # Rest of your existing showMap code, but use `board` instead of `mainBoard`
    terrain = board.terrain
    # ... etc
```

Does this approach make sense? Should I show the redirect logic for `hex_clicked` and `country_hex_clicked` too?